In [1]:
!pip install -q pypdf sentence-transformers faiss-cpu google-genai gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 19.2 MB/s eta 0:00:00


# Technical Documentation Assistant Using RAG

### A Retrieval-Augmented Generation Based Question Answering System

**Technology:** Python  
**Domain:** Generative AI / RAG  
**Tools:** Python, FAISS, Sentence Transformers, Gemini, Gradio

## 1. Introduction

This project develops a Technical Documentation Assistant using
Retrieval-Augmented Generation (RAG). The system allows users to ask
questions about technical documentation and provides answers based
only on the information retrieved from the documentation.

## 2. Problem Statement

Technical documentation can contain a large amount of information,
making it difficult for users to quickly find the required information.

The proposed system provides a chatbot that retrieves relevant
information from technical documentation and generates a
context-grounded answer for the user's question.

## 3. Objectives

- To create a technical documentation knowledge base.
- To extract and process documentation from PDF files.
- To divide documentation into meaningful chunks.
- To generate embeddings for documentation.
- To implement semantic search using FAISS.
- To retrieve relevant documentation for a user query.
- To generate answers using a Large Language Model.
- To provide documentation references with answers.
- To prevent answers based on information outside the knowledge base.

In [2]:
import os
import re
import numpy as np

from pathlib import Path
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss

In [33]:
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
from google.api_core.exceptions import ServerError


In [5]:
import os

os.listdir("/content")

['.config', '.ipynb_checkpoints', 'python_documentation.pdf', 'sample_data']

## 6. Upload Technical Documentation

The technical documentation PDF is uploaded to Google Colab.
The document is used as the knowledge source for the RAG system.

In [6]:
from pypdf import PdfReader

pdf_path = "/content/python_documentation.pdf"

reader = PdfReader(pdf_path)

print("Number of pages:", len(reader.pages))

Number of pages: 4


## 7. PDF Text Extraction

The uploaded PDF is processed using PyPDF. Text is extracted
page by page so that the original page number can be retained
for documentation references.

In [7]:
pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text()

    if text:
        text = re.sub(r"\s+", " ", text).strip()

        pages.append({
            "page": page_number,
            "text": text
        })

print("Pages with text:", len(pages))

Pages with text: 4


In [8]:
print(pages[0]["text"][:2000])

Python support for free threading Release 3.14.0rc3 Guido van Rossum and the Python development team October 01, 2025 Python Software Foundation Email: docs@python.org Contents 1 Installation 2 2 Identifying free-threaded Python 2 3 The global interpreter lock in free-threaded Python 2 4 Thread safety 2 5 Known limitations 2 5.1 Immortalization . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 3 5.2 Frame objects . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 3 5.3 Iterators . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 3 5.4 Single-threaded performance . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 3 6 Behavioral changes 3 6.1 Context variables . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 3 6.2 Warning filters . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 

## 8. Text Chunking

Large documents are divided into smaller text chunks.

Chunking helps the retrieval system find the specific sections
that are relevant to a user's question.

In [9]:

#Chunk the documentation
def create_chunks(text, chunk_size=500, overlap=100):
    words = text.split()

    chunks = []

    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))

        chunk = " ".join(words[start:end])

        chunks.append(chunk)

        if end == len(words):
            break

        start = end - overlap

    return chunks

In [10]:
test_chunks = create_chunks(pages[0]["text"])

print("Number of chunks:", len(test_chunks))
print(test_chunks[0][:1000])

Number of chunks: 2
Python support for free threading Release 3.14.0rc3 Guido van Rossum and the Python development team October 01, 2025 Python Software Foundation Email: docs@python.org Contents 1 Installation 2 2 Identifying free-threaded Python 2 3 The global interpreter lock in free-threaded Python 2 4 Thread safety 2 5 Known limitations 2 5.1 Immortalization . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 3 5.2 Frame objects . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 3 5.3 Iterators . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 3 5.4 Single-threaded performance . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 3 6 Behavioral changes 3 6.1 Context variables . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 3 6.2 Warning filters . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 

## 9. Documentation Knowledge Base

The extracted and chunked documentation forms the knowledge base
of the Technical Documentation Assistant.

Each chunk retains its page number so that the system can later
provide documentation references.

In [11]:
documents = []

for page in pages:

    chunks = create_chunks(page["text"])

    for chunk in chunks:

        documents.append({
            "page": page["page"],
            "text": chunk
        })

print("Total documentation chunks:", len(documents))

Total documentation chunks: 5


In [12]:
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
texts = [doc["text"] for doc in documents]

## 10. Embedding Generation

Each documentation chunk is converted into a numerical vector
using a Sentence Transformer model.

These vectors allow the system to compare the meaning of the
user's question with the meaning of documentation chunks.

In [14]:
#embeddings
embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Embedding shape: (5, 384)


## 11. FAISS Vector Database

FAISS is used to store the documentation embeddings and perform
similarity search.

When a user asks a question, its embedding is compared with
the stored documentation embeddings.

In [15]:
#Create FAISS vector database
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings.astype("float32"))

print("Vectors stored:", index.ntotal)

Vectors stored: 5


## 12. Semantic Search

Semantic search retrieves documentation based on the meaning
of the user's question rather than only matching exact words.

In [17]:
#Create semantic search
def search_documents(query, top_k=4):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        results.append({
            "score": float(score),
            "page": documents[idx]["page"],
            "text": documents[idx]["text"]
        })

    return results

## 13. Testing Semantic Retrieval

In [18]:
#Test semantic retrieval

questions = [
    "What is a Python list?",
    "How are functions created?",
    "What is exception handling?",
    "What is a dictionary?"
]

for question in questions:

    print("\nQUESTION:", question)

    results = search_documents(question, top_k=2)

    for result in results:
        print(
            "Page:",
            result["page"],
            "| Score:",
            round(result["score"], 3)
        )


QUESTION: What is a Python list?
Page: 4 | Score: 0.43
Page: 1 | Score: 0.354

QUESTION: How are functions created?
Page: 3 | Score: 0.112
Page: 1 | Score: 0.084

QUESTION: What is exception handling?
Page: 3 | Score: 0.154
Page: 1 | Score: 0.123

QUESTION: What is a dictionary?
Page: 4 | Score: 0.135
Page: 1 | Score: 0.111


## 14. Gemini LLM Configuration

The retrieved documentation is supplied to the Gemini language
model as context. The model is instructed to answer only from
the retrieved documentation.

In [19]:
from getpass import getpass

GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


In [20]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

## 15. RAG Prompt and Grounding

The prompt instructs the LLM to use only the retrieved
documentation as evidence.

If the required information is not available, the system
returns an information-not-found response.

In [21]:
def create_prompt(question, results):

    context = ""

    for i, result in enumerate(results, start=1):

        context += f"""
SOURCE {i}
Page: {result['page']}

{result['text']}

-------------------------
"""

    prompt = f"""
You are a Technical Documentation Assistant.

Answer the user's question ONLY using the documentation
provided below.

Do not use outside knowledge.

If the answer cannot be found in the documentation,
respond exactly:

Information not found in the provided documentation.

Do not invent information.

DOCUMENTATION:
{context}

USER QUESTION:
{question}

ANSWER:
"""

    return prompt

## 16. Answer Generation

The retrieved documentation is passed to Gemini along with
the user's question. Gemini generates a context-grounded answer.

In [24]:
def generate_answer(question, top_k=4):

    results = search_documents(question, top_k)

    if not results:
        return "Information not found in the provided documentation.", []

    if results[0]["score"] < 0.20:
        return "Information not found in the provided documentation.", results

    prompt = create_prompt(question, results)

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text, results

In [34]:
@retry(
    wait=wait_exponential(multiplier=1, min=4, max=10),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type(ServerError),
    reraise=True
)
def generate_answer_with_retry(question, top_k=4):
    return generate_answer(question, top_k)

I've modified the `generate_answer` function to `generate_answer_with_retry` which includes retry logic using the `tenacity` library. This will automatically retry the function up to 5 times with an exponential backoff if a `ServerError` (like a 503 error) occurs.

## 17. Documentation References

The system displays the page numbers of the documentation
used to retrieve information for the answer.

In [35]:
question = "Identifying free-threaded Python"

answer, sources = generate_answer_with_retry(question)

print("ANSWER:")
print(answer)

print("\nSOURCES:")

for source in sources:
    print(
        "Page:",
        source["page"],
        "Score:",
        round(source["score"], 3)
    )

ANSWER:
To identify if the current Python interpreter supports free threading, you can use the following methods:

* **Check version strings:** `python -VV` and `sys.version` will contain the text `"free-threading build"`.
* **Check if the GIL is disabled:** The `sys._is_gil_enabled()` function can be used to check whether the GIL is actually disabled in the running process.
* **Check configuration variables:** The configuration variable `sysconfig.get_config_var("Py_GIL_DISABLED")` can be used to determine whether the build supports free threading. If set to `1`, the build supports free threading. This is the recommended mechanism for decisions related to build configuration.

SOURCES:
Page: 1 Score: 0.72
Page: 1 Score: 0.715
Page: 2 Score: 0.659
Page: 3 Score: 0.457


## 18. Out-of-Knowledge-Base Testing

The system is tested with questions whose answers are not
available in the technical documentation.

The expected behavior is to report that the information
was not found instead of generating an unsupported answer.

In [26]:
question = "Who is the current Prime Minister of India?"

answer, sources = generate_answer(question)

print(answer)

Information not found in the provided documentation.


In [27]:
!pip install -q gradio

## 19. Chatbot Interface

A simple Gradio interface is used to allow users to interact
with the Technical Documentation Assistant.

In [28]:
import gradio as gr

def chatbot(question):

    if not question.strip():
        return "Please enter a question.", ""

    answer, sources = generate_answer(question)

    source_text = "\n".join(
        [
            f"Page {s['page']} | Similarity: {s['score']:.3f}"
            for s in sources
        ]
    )

    return answer, source_text

In [29]:
demo = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(
        label="Ask about the documentation",
        placeholder="Example: What is a Python list?"
    ),
    outputs=[
        gr.Textbox(label="Answer"),
        gr.Textbox(label="Documentation References")
    ],
    title="Technical Documentation Assistant",
    description="Ask questions about the uploaded technical documentation."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a6c668a1f866b2cb0b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 22. Conclusion

The Technical Documentation Assistant demonstrates the use of
Retrieval-Augmented Generation for question answering over
technical documentation.

The system combines document processing, embeddings, semantic
retrieval, vector search and an LLM to generate context-grounded
answers. Documentation references help users identify the source
of the retrieved information.